In [ ]:
# Install dependencies
!pip install fairlearn aif360 kagglehub reportlab tensorflow matplotlib seaborn scikit-learn --quiet

import os
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models, applications, optimizers
from sklearn.model_selection import train_test_split
from fairlearn.metrics import MetricFrame, selection_rate, false_positive_rate
from fairlearn.postprocessing import ThresholdOptimizer
from fairlearn.reductions import ExponentiatedGradient, DemographicParity
from sklearn.linear_model import LogisticRegression
from sklearn.utils import resample
import kagglehub
import matplotlib.image as mpimg
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, accuracy_score
from sklearn.preprocessing import label_binarize
import itertools

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import shutil
import os
import kagglehub

# Download dataset
raw_path = kagglehub.dataset_download("aibloy/fairface")

local_data_dir = "data/fairface"
if os.path.exists(local_data_dir):
    shutil.rmtree(local_data_dir)
os.makedirs(local_data_dir, exist_ok=True)

# Copy all files/folders from raw_path to local_data_dir
for f in os.listdir(raw_path):
    src = os.path.join(raw_path, f)
    dst = os.path.join(local_data_dir, f)
    if os.path.isdir(src):
        shutil.copytree(src, dst)
    else:
        shutil.copy2(src, dst)

# Define paths
TRAIN_IMG_DIR = os.path.join(local_data_dir, "FairFace", "train")
VAL_IMG_DIR   = os.path.join(local_data_dir, "FairFace", "val")
TRAIN_CSV     = os.path.join(local_data_dir, "FairFace", "train_labels.csv")
VAL_CSV       = os.path.join(local_data_dir, "FairFace", "val_labels.csv")

# Load CSVs
df_train = pd.read_csv(TRAIN_CSV)
df_val   = pd.read_csv(VAL_CSV)

print(f"Train images folder: {TRAIN_IMG_DIR}")
print(f"Validation images folder: {VAL_IMG_DIR}")
print(f"Train CSV shape: {df_train.shape}")
print(f"Val CSV shape: {df_val.shape}")


In [ ]:
# EDA plots
plt.figure(figsize=(10,5))
sns.countplot(data=df_train, x="race", hue="gender")
plt.xticks(rotation=45)
plt.title("Race × Gender Distribution (Train)")
plt.tight_layout()
plt.savefig("eda_distribution.png")
plt.show()

In [ ]:
# Ensure filenames only
df_train['file'] = df_train['file'].apply(lambda x: os.path.basename(str(x)))

# List actual files in the train directory
available_files = set(os.listdir(TRAIN_IMG_DIR))

samples_per_race = 2
races = df_train['race'].unique()

plt.figure(figsize=(samples_per_race*2, len(races)*2.5))

for i, race in enumerate(races):
    # Filter only files that exist in the folder
    race_files = df_train[df_train['race'] == race]['file']
    race_files = race_files[race_files.isin(available_files)]

    # If no valid files for this race, skip
    if race_files.empty:
        continue

    # Sample images
    sample_files = race_files.sample(min(samples_per_race, len(race_files)), random_state=42)

    for j, fname in enumerate(sample_files):
        img_path = os.path.join(TRAIN_IMG_DIR, fname)

        if not os.path.exists(img_path):
            continue

        img = mpimg.imread(img_path)

        plt.subplot(len(races), samples_per_race, i*samples_per_race + j + 1)
        plt.imshow(img)
        plt.axis('off')
        plt.title(race, fontsize=8)

plt.suptitle("Sample Images by Race", fontsize=16)
plt.tight_layout()
plt.savefig("sample_images_by_race.png")
plt.show()


In [ ]:
# Data Generators
IMG_SIZE = 224
BATCH_SIZE = 32
train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255, horizontal_flip=True, validation_split=0.1
)

In [ ]:
# Fix file paths for train and val
df_train['file'] = df_train['file'].apply(lambda x: x.strip())
df_val['file']   = df_val['file'].apply(lambda x: x.strip())

# If CSV contains full path, keep only filename
df_train['file'] = df_train['file'].apply(lambda x: os.path.basename(x))
df_val['file']   = df_val['file'].apply(lambda x: os.path.basename(x))

# Verify sample
print(df_train['file'].head())

# Now load generators
train_gen = train_datagen.flow_from_dataframe(
    dataframe=df_train,
    directory=TRAIN_IMG_DIR,
    x_col="file",
    y_col="gender",
    subset="training",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=True
)
val_gen = train_datagen.flow_from_dataframe(
    dataframe=df_train,
    directory=TRAIN_IMG_DIR,
    x_col="file",
    y_col="gender",
    subset="validation",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False
)


#Base Model Approach

In [ ]:
base_model_race = applications.ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
x = layers.GlobalAveragePooling2D()(base_model_race.output)
output = layers.Dense(1, activation='sigmoid')(x)
model_race = models.Model(inputs=base_model_race.input, outputs=output)

model_race.compile(
    optimizer=optimizers.Adam(1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# === Train ===
history_race = model_race.fit(
    train_gen,
    validation_data=val_gen,
    epochs=5
)

In [ ]:
# Accuracy plot
plt.figure()
plt.plot(history_race.history['accuracy'], label='Train Accuracy')
plt.plot(history_race.history['val_accuracy'], label='Validation Accuracy')
plt.title("Baseline Training Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.savefig("baseline_accuracy.png")
plt.show()

# Loss plot
plt.figure()
plt.plot(history_race.history['loss'], label='Train Loss')
plt.plot(history_race.history['val_loss'], label='Validation Loss')
plt.title("Baseline Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.savefig("baseline_loss.png")
plt.show()


In [ ]:
# Save the trained model
model_race.save("model_baseline.h5")
print("Model saved as model_baseline.h5")

In [ ]:
# === Predictions (Binary) ===
y_scores = model_race.predict(val_gen).flatten()   # shape (N,)
y_pred = (y_scores > 0.5).astype(int)              # threshold at 0.5
y_true = val_gen.classes                           # already 0 or 1
class_names = list(val_gen.class_indices.keys())   # e.g., ["Female", "Male"]

# === Classification Report ===
print("\nClassification Report (Binary):")
print(classification_report(y_true, y_pred, target_names=class_names))

# === Confusion Matrix ===
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.title("Confusion Matrix - Binary Classification")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.savefig("cm_binary.png")
plt.show()

# === ROC & AUC (Binary) ===
fpr, tpr, _ = roc_curve(y_true, y_scores)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f"ROC curve (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Binary Classification")
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("roc_curve_binary.png")
plt.show()



In [ ]:
# Get sensitive feature (race) for each validation sample
sensitive_features = df_val.iloc[val_gen.index_array]["race"].values

# Create DataFrame
val_df_race = pd.DataFrame({
    "y_true": y_true,            # binary ground truth
    "y_pred": y_pred,            # binary predictions
    "race": sensitive_features   # actual race from dataset
})

# Fairness metrics by race
metric_frame = MetricFrame(
    metrics={
        "accuracy": accuracy_score,
        "selection_rate": selection_rate,
        "fpr": false_positive_rate
    },
    y_true=val_df_race["y_true"],
    y_pred=val_df_race["y_pred"],
    sensitive_features=val_df_race["race"]
)

print("\n=== Fairness Metrics by Race ===")
print(metric_frame.by_group)

# Disparate Impact Ratio
groups = metric_frame.by_group.index.tolist()
if len(groups) >= 2:
    acc_group_a = metric_frame.by_group.loc[groups[0], "accuracy"]
    acc_group_b = metric_frame.by_group.loc[groups[1], "accuracy"]
    disparate_impact_ratio = acc_group_a / acc_group_b if acc_group_b != 0 else None
    print(f"\nDisparate Impact Ratio ({groups[0]} vs {groups[1]}): {disparate_impact_ratio:.3f}")

# Equal Opportunity Difference
if len(groups) >= 2:
    fpr_group_a = metric_frame.by_group.loc[groups[0], "fpr"]
    fpr_group_b = metric_frame.by_group.loc[groups[1], "fpr"]
    equal_opportunity_diff = fpr_group_a - fpr_group_b
    print(f"Equal Opportunity Difference ({groups[0]} - {groups[1]}): {equal_opportunity_diff:.3f}")

In [ ]:
# --- Binary Evaluation Function with Race ---
def evaluate_model_binary(y_true, y_scores, class_names, race_features, title_prefix, save_prefix):
    """
    y_true: Ground truth labels (0/1)
    y_scores: Predicted probabilities
    class_names: ["Female", "Male"] or similar
    race_features: List/array of race labels for each sample
    title_prefix: For plot titles
    save_prefix: For saved file names
    """
    # Binary predictions
    y_pred = (y_scores.flatten() > 0.5).astype(int)

    # Classification Report
    print(f"\nClassification Report - {title_prefix}")
    print(classification_report(y_true, y_pred, target_names=class_names))

    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f"{title_prefix} - Confusion Matrix")
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.tight_layout()
    plt.savefig(f"{save_prefix}_cm.png")
    plt.show()

    # ROC–AUC (binary)
    fpr, tpr, _ = roc_curve(y_true, y_scores)
    roc_auc = auc(fpr, tpr)
    plt.figure(figsize=(6,5))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f"AUC = {roc_auc:.3f}")
    plt.plot([0, 1], [0, 1], 'k--', lw=2)
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"{title_prefix} - ROC Curve")
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(f"{save_prefix}_roc.png")
    plt.show()

    # === Fairness Metrics by Race ===
    val_df_eval = pd.DataFrame({
        "y_true": y_true,
        "y_pred": y_pred,
        "race": race_features
    })
    metric_frame = MetricFrame(
        metrics={
            "accuracy": accuracy_score,
            "selection_rate": selection_rate,
            "fpr": false_positive_rate
        },
        y_true=val_df_eval["y_true"],
        y_pred=val_df_eval["y_pred"],
        sensitive_features=val_df_eval["race"]
    )

    print("\n=== Fairness Metrics by Race ===")
    print(metric_frame.by_group)

    # Disparate Impact Ratio & Equal Opportunity Difference (first 2 races)
    groups = metric_frame.by_group.index.tolist()
    if len(groups) >= 2:
        acc_ratio = metric_frame.by_group.loc[groups[0], "accuracy"] / metric_frame.by_group.loc[groups[1], "accuracy"]
        eq_diff = metric_frame.by_group.loc[groups[0], "fpr"] - metric_frame.by_group.loc[groups[1], "fpr"]
        print(f"\nDisparate Impact Ratio ({groups[0]} vs {groups[1]}): {acc_ratio:.3f}")
        print(f"Equal Opportunity Difference ({groups[0]} - {groups[1]}): {eq_diff:.3f}")


# Pre-processing Approach

In [ ]:
df_balanced = pd.concat([
    resample(group, replace=True, n_samples=df_train['race'].value_counts().max(), random_state=42)
    for _, group in df_train.groupby('race')
])

train_gen_bal = train_datagen.flow_from_dataframe(
    dataframe=df_balanced,
    directory=TRAIN_IMG_DIR,
    x_col="file",
    y_col="gender",
    subset="training",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=True
)

val_gen_bal = train_datagen.flow_from_dataframe(
    dataframe=df_balanced,
    directory=TRAIN_IMG_DIR,
    x_col="file",
    y_col="gender",
    subset="validation",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False
)

model_pre = tf.keras.models.clone_model(model_race)
model_pre.compile(optimizer=optimizers.Adam(1e-4), loss='binary_crossentropy', metrics=['accuracy'])
model_pre.fit(train_gen_bal, validation_data=val_gen, epochs=3)

In [ ]:
# Save the trained model
model_pre.save("model_pre.h5")
print("Model saved as model_pre.h5")

In [ ]:
y_scores_pre = model_pre.predict(val_gen)
evaluate_model_binary(val_gen.classes, y_scores_pre, list(val_gen.class_indices.keys()),
                      sensitive_features=[df_train.iloc[i]["race"] for i in val_gen.index_array],
                      title_prefix="Pre-processing", save_prefix="preproc")

# In-processing Approach

In [ ]:
feature_extractor = tf.keras.Model(inputs=model_race.input, outputs=model_race.layers[-2].output)
X_train_feats = feature_extractor.predict(train_gen)
y_train_labels = train_gen.classes
sensitive_train = [df_train.iloc[i]["race"] for i in train_gen.index_array]

X_val_feats = feature_extractor.predict(val_gen)
y_val_labels = val_gen.classes
sensitive_val = [df_train.iloc[i]["race"] for i in val_gen.index_array]

clf = LogisticRegression(max_iter=10)
mitigator = ExponentiatedGradient(clf, constraints=DemographicParity())
mitigator.fit(X_train_feats, y_train_labels, sensitive_features=sensitive_train)

y_pred_inproc = mitigator.predict(X_val_feats)
y_scores_inproc = np.array(y_pred_inproc)  # logistic regression outputs predicted labels only
evaluate_model_binary(y_val_labels, y_scores_inproc, list(val_gen.class_indices.keys()),
                      sensitive_features=sensitive_val,
                      title_prefix="In-processing", save_prefix="inproc")



In [ ]:
import joblib

# Save the fairness-aware logistic regression model
joblib.dump(mitigator, "inprocessing_model.pkl")
print("In-processing fairness-aware model saved as inprocessing_model.pkl")

# Post-processing Approach

In [ ]:


# === Evaluation function expecting race_features ===
def evaluate_model_binary(y_true, y_scores, class_names, race_features, title_prefix, save_prefix):
    y_pred = (y_scores.flatten() > 0.5).astype(int)

    # Classification report
    print(f"\nClassification Report - {title_prefix}")
    print(classification_report(y_true, y_pred, target_names=class_names))

    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f"{title_prefix} - Confusion Matrix")
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.tight_layout()
    plt.savefig(f"{save_prefix}_cm.png")
    plt.show()

    # ROC–AUC
    try:
        fpr, tpr, _ = roc_curve(y_true, y_scores)
        roc_auc = auc(fpr, tpr)
        plt.figure(figsize=(6,5))
        plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
        plt.plot([0,1], [0,1], 'k--')
        plt.title(f"{title_prefix} - ROC Curve")
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.legend(loc="lower right")
        plt.tight_layout()
        plt.savefig(f"{save_prefix}_roc.png")
        plt.show()
    except Exception as e:
        print("ROC/AUC plotting skipped:", e)
        roc_auc = None

    # Fairness metrics by race
    val_df_eval = pd.DataFrame({
        "y_true": y_true,
        "y_pred": y_pred,
        "race": race_features
    })
    metric_frame = MetricFrame(
        metrics={"accuracy": accuracy_score, "selection_rate": selection_rate, "fpr": false_positive_rate},
        y_true=val_df_eval["y_true"],
        y_pred=val_df_eval["y_pred"],
        sensitive_features=val_df_eval["race"]
    )
    print(f"\n=== Fairness Metrics by Race - {title_prefix} ===")
    print(metric_frame.by_group)

    groups = metric_frame.by_group.index.tolist()
    if len(groups) >= 2:
        acc_ratio = metric_frame.by_group.loc[groups[0], "accuracy"] / metric_frame.by_group.loc[groups[1], "accuracy"]
        eq_diff = metric_frame.by_group.loc[groups[0], "fpr"] - metric_frame.by_group.loc[groups[1], "fpr"]
        print(f"\nDisparate Impact Ratio ({groups[0]} vs {groups[1]}): {acc_ratio:.3f}")
        print(f"Equal Opportunity Difference ({groups[0]} - {groups[1]}): {eq_diff:.3f}")
    return {
        "Model": title_prefix,
        "Accuracy": accuracy_score(y_true, y_pred),
        "AUC": roc_auc,
        "Disparate Impact Ratio": acc_ratio if len(groups) >=2 else None,
        "Equal Opportunity Diff": eq_diff if len(groups) >=2 else None
    }

# === Post-processing using ThresholdOptimizer on score ===
# Prepare validation data
X_val_scores = model_race.predict(val_gen).flatten()  # original probability
X_val_feat = X_val_scores.reshape(-1, 1)  # single-feature matrix
y_val = val_gen.classes
race_features_val = df_val.iloc[val_gen.index_array]["race"].values

# Wrapper that exposes predict_proba interface
class ScoreProbWrapper:
    def fit(self, X, y=None):
        return self
    def predict_proba(self, X):
        probs = X.flatten()
        return np.vstack([1 - probs, probs]).T
    def predict(self, X):
        probs = X.flatten()
        return (probs > 0.5).astype(int)

identity_estimator = ScoreProbWrapper()

threshold_optimizer = ThresholdOptimizer(
    estimator=identity_estimator,
    constraints="demographic_parity",
    prefit=True
)

threshold_optimizer.fit(X_val_feat, y_val, sensitive_features=race_features_val)

# Get post-processed binary predictions (requires sensitive_features)
y_pred_post = threshold_optimizer.predict(X_val_feat, sensitive_features=race_features_val)

# Try to extract scores (best effort)
try:
    y_scores_post = threshold_optimizer._pmf_predict(X_val_feat, sensitive_features=race_features_val)[:, 1]
except Exception:
    # fallback to raw score adjusted (or binary)
    y_scores_post = y_pred_post.astype(float)

# Evaluate post-processing
metrics_post = evaluate_model_binary(
    y_true=y_val,
    y_scores=y_scores_post,
    class_names=list(val_gen.class_indices.keys()),
    race_features=race_features_val,
    title_prefix="Post-processing",
    save_prefix="postproc"
)


In [ ]:

# Save the post-processing fairness model
joblib.dump(threshold_optimizer, "postprocessing_model.pkl")
print("Post-processing model saved as postprocessing_model.pkl")

# Comparsion of All models approaches

In [ ]:
# === Collect results for all models ===
results = []

# Base model
results.append(evaluate_model_binary(
    val_gen.classes, model_race.predict(val_gen),
    list(val_gen.class_indices.keys()),
    race_features=df_val.iloc[val_gen.index_array]["race"].values,
    title_prefix="Base Model", save_prefix="base"
))

# Pre-processing
results.append(evaluate_model_binary(
    val_gen.classes, model_pre.predict(val_gen),
    list(val_gen.class_indices.keys()),
    race_features=df_val.iloc[val_gen.index_array]["race"].values,
    title_prefix="Pre-processing", save_prefix="preproc"
))

# In-processing
results.append(evaluate_model_binary(
    y_val_labels, y_scores_inproc,
    list(val_gen.class_indices.keys()),
    race_features=sensitive_val,
    title_prefix="In-processing", save_prefix="inproc"
))

# Post-processing
results.append(evaluate_model_binary(
    val_gen.classes, y_scores_post,
    list(val_gen.class_indices.keys()),
    race_features=df_val.iloc[val_gen.index_array]["race"].values,
    title_prefix="Post-processing", save_prefix="postproc"
))

# === Create Comparison DataFrame ===
df_results = pd.DataFrame(results)
print("\n=== Model Comparison ===")
print(df_results)

# === Plot Accuracy and AUC comparison ===
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

# Accuracy Plot
sns.barplot(x="Model", y="Accuracy", data=df_results, ax=ax[0])
ax[0].set_title("Accuracy Comparison")
ax[0].set_ylim(0, 1)

# AUC Plot
sns.barplot(x="Model", y="AUC", data=df_results, ax=ax[1])
ax[1].set_title("AUC Comparison")
ax[1].set_ylim(0, 1)

plt.tight_layout()
plt.savefig("model_comparison.png")
plt.show()